# The Probabilistic Mindset: Why Deterministic Thinking Breaks

**Phase 00** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-00/04-probabilistic-mindset.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your AI feature passes all tests on Monday. By Thursday, three users report it returning wrong answers. You pull the logs, find one of the bad traces, re-run it manually, and it works fine. You close the bug report: "Cannot reproduce."

Two weeks later you get five more reports. You re-run again. Fine again. The issue is not in any single trace. The issue is in the distribution: 8% of calls return a response that does not match your schema, 3% return the wrong category, and 2% return an empty string for a field that should always be populated. None of these are bugs in your code. They are properties of the distribution your model samples from.

Engineers who treat models as deterministic fun...

## The Concept

### Deterministic Function vs. Probabilistic Sampler

```
DETERMINISTIC FUNCTION (what software engineers expect):

Input A ----[ function ]----> Output X (always)
Input A ----[ function ]----> Output X (always)
Input A ----[ function ]----> Output X (always)

Examples: hash(), sort(), parseInt()


PROBABILISTIC SAMPLER (what an LLM actually is):

Input A ----[ model ]----> Output X  (60% of the time)
Input A ----[ model ]----> Output Y  (25% of the time)
Input A ----[ model ]----> Output Z  (10% of the time)
Input A ----[ model ]----> Output W  ( 5% of the time)

The output is a SAMPLE from a...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 04: The Probabilistic Mindset

Demonstrates:
- Running the same prompt N times to observe output variance
- Measuring the distribution of outputs across runs
- Comparing temperature settings (0.0 vs 1.0) on variance
- The robust_classify pattern that handles output variations
- Why single-trace testing is insufficient for AI systems

Run with: uv run python main.py
Note: This makes ~40 API calls. Estimated cost: <$0.01 using Haiku.
"""

import os
from collections import Counter
from dotenv import load_dotenv
import anthropic


load_dotenv()


def run_n_times(
    client: anthropic.Anthropic,
    prompt: str,
    n: int = 10,
    temperature: float = 1.0,
    max_tokens: int = 32,
) -> list[str]:
    """Run the same prompt N times and return all raw outputs."""
    results = []
    for i in range(n):
        response = client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.content[0].text.strip()
        results.append(text)
        print(f"  Run {i+1:2d}: {text!r}")
    return results


def measure_distribution(
    client: anthropic.Anthropic,
    prompt: str,
    n: int = 15,
    temperature: float = 1.0,
) -> dict:
    """
    Run a prompt N times and print a distribution histogram.
    Returns stats dict with consistency_pct (fraction that matched the top output).
    """
    print(f"\nRunning {n} times at temperature={temperature}:")
    results = run_n_times(client, prompt, n, temperature)

    counter = Counter(results)
    total = len(results)

    print(f"\nDistribution ({total} runs):")
    for output, count in counter.most_common():
        pct = count / total * 100
        bar = "#" * int(pct / 5)
        print(f"  {output!r:35} {count:3}x ({pct:5.1f}%) {bar}")

    most_common_output, most_common_count = counter.most_common(1)[0]
    consistency_pct = most_common_count / total * 100

    print(f"\nSummary:")
    print(f"  Unique outputs:        {len(counter)}")
    print(f"  Most common output:    {most_common_output!r}")
    print(f"  Consistency:           {consistency_pct:.1f}%")

    return {
        "outputs": results,
        "distribution": dict(counter),
        "unique_count": len(counter),
        "consistency_pct": consistency_pct,
        "most_common": most_common_output,
    }


def compare_temperatures(
    client: anthropic.Anthropic,
    prompt: str,
    temperatures: list[float] = [0.0, 0.5, 1.0],
    n_per_temp: int = 8,
) -> None:
    """Show how temperature affects output variance for the same prompt."""
    print(f"\n=== Temperature Variance Comparison ===")
    print(f"Prompt: '{prompt[:70]}'")
    print(f"Runs per temperature: {n_per_temp}\n")

    for temp in temperatures:
        results = run_n_times(client, prompt, n_per_temp, temp)
        unique = len(set(r.lower().strip(".,!?") for r in results))
        consistency = Counter(results).most_common(1)[0][1] / n_per_temp * 100
        print(f"  Temperature {temp}: {unique}/{n_per_temp} unique, {consistency:.0f}% consistency\n")


def robust_classify(client: anthropic.Anthropic, text: str) -> str:
    """
    Classify text as POSITIVE, NEGATIVE, or NEUTRAL.

    Uses temperature=0.0 for minimal variance on a classification task.
    Normalizes the output to handle "Positive", "POSITIVE", "positive.", etc.
    Returns "UNKNOWN" and logs when the model produces an unexpected format.

    In production: log UNKNOWN outputs and review them to improve your prompt.
    """
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=16,
        temperature=0.0,
        messages=[{
            "role": "user",
            "content": (
                "Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL. "
                "Reply with exactly one word: POSITIVE, NEGATIVE, or NEUTRAL.\n\n"
                f"Text: {text}"
            ),
        }],
    )
    raw = response.content[0].text.strip().upper()

    # Normalize variations
    if "POSITIVE" in raw:
        return "POSITIVE"
    if "NEGATIVE" in raw:
        return "NEGATIVE"
    if "NEUTRAL" in raw:
        return "NEUTRAL"

    # Unexpected output -- log in production, return safe default
    print(f"WARNING: unexpected classification output: {raw!r} for text: {text!r}")
    return "UNKNOWN"


def demonstrate_brittle_vs_robust(client: anthropic.Anthropic) -> None:
    """
    Show the failure mode: brittle exact string matching vs. robust normalization.
    """
    print("\n=== Brittle vs. Robust Output Handling ===")

    test_texts = [
        "The product works perfectly.",
        "Absolutely terrible experience.",
        "It was okay, nothing special.",
    ]

    for text in test_texts:
        # Run 3 times to show natural variation
        raw_outputs = []
        for _ in range(3):
            r = client.messages.create(
                model="claude-3-5-haiku-20241022",
                max_tokens=16,
                temperature=0.7,  # some variance to show variation
                messages=[{
                    "role": "user",
                    "content": f"Is this positive, negative, or neutral? Just the word.\nText: {text}",
                }],
            )
            raw_outputs.append(r.content[0].text.strip())

        print(f"\nText: {text!r}")
        print(f"  Raw outputs: {raw_outputs}")

        # BRITTLE: exact match would fail on most of these
        brittle_results = [o == "positive" for o in raw_outputs]
        brittle_pass_rate = sum(brittle_results) / len(brittle_results) * 100
        print(f"  Brittle (exact match 'positive'): {sum(brittle_results)}/{len(brittle_results)} pass ({brittle_pass_rate:.0f}%)")

        # ROBUST: normalization handles variants
        robust_result = robust_classify(client, text)
        print(f"  Robust (normalized):              {robust_result}")


def main() -> None:
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("ERROR: ANTHROPIC_API_KEY not set.")
        return

    client = anthropic.Anthropic(api_key=api_key)

    print("=== Lesson 04: The Probabilistic Mindset ===")
    print("This script makes ~40 API calls. Estimated cost: <$0.01 (Haiku)\n")

    # 1. Observe variance with a simple question
    print("--- Part 1: Observe Output Variance (temperature=1.0) ---")
    prompt = "Is 'The meeting was fine' POSITIVE, NEGATIVE, or NEUTRAL? One word only."
    stats = measure_distribution(client, prompt, n=10, temperature=1.0)

    # 2. Compare temperatures
    print("\n--- Part 2: Temperature Effect on Variance ---")
    compare_temperatures(client, prompt, [0.0, 1.0], n_per_temp=5)

    # 3. Brittle vs. robust handling
    print("\n--- Part 3: Brittle vs. Robust Output Handling ---")
    demonstrate_brittle_vs_robust(client)

    # 4. Summary
    print("\n=== Summary ===")
    print(f"In Part 1, the prompt generated {stats['unique_count']} unique outputs.")
    print(f"Consistency at temperature=1.0: {stats['consistency_pct']:.0f}%")
    print()
    print("Key takeaways:")
    print("1. The same prompt does not always produce the same output.")
    print("2. Temperature=0.0 minimizes variance for structured tasks.")
    print("3. Always normalize model output before comparing -- never exact-match.")
    print("4. One test run tells you nothing about your system's failure rate.")
    print("5. Eval N outputs, not 1.")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What is the critical flaw in this testing approach, and what should it be replaced with?**

- A. The test should use a different assertion library
- B. A single-run assertion only captures one sample from the output distribution; it should run the classification 20+ times and assert a minimum pass rate
- C. The test should use temperature=0.0 to make the result deterministic
- D. The test text is too short to produce a reliable classification

**2. What does 'cannot reproduce' actually indicate in this situation?**

- A. The bug was transient and has fixed itself
- B. The failing log was corrupted and does not represent a real failure
- C. The failure is a property of the output distribution -- it occurs at some rate, not deterministically, so a single re-run is unlikely to reproduce it
- D. The model was updated between the production run and your manual re-run

**3. What is the correct pattern to fix this?**

- A. Set temperature=0.0 to make the model always return exactly 'CONTRACT'
- B. Normalize the output before comparing: `if 'CONTRACT' in response.content[0].text.upper(): route_to_contracts()`
- C. Add all possible variants to the if/else: `if response == 'CONTRACT' or response == 'Contract'...`
- D. Use a different model that follows formatting instructions more reliably

**4. What temperature setting is appropriate here, and why?**

- A. Keep temperature=0.0 because it produces the highest quality outputs
- B. Set temperature=0.7-0.9 because higher variance produces more creative variation between regenerations
- C. Set temperature=1.5 to maximize diversity
- D. Temperature does not affect creative writing quality

**5. What was the most likely flaw in the 10-question evaluation?**

- A. The evaluation was done at temperature=1.0 instead of 0.0
- B. 10 examples is too small to estimate production accuracy reliably, and manually selected examples likely did not capture the tail of difficult or unusual real user queries
- C. The model was updated between evaluation and launch, changing its behavior
- D. 90% accuracy on a Q&A system is an inherently unreliable metric

**6. What is the correct diagnostic approach?**

- A. Run it 3 more times; if it passes 2 out of 3, it is working well enough
- B. Run the failing input 20-50 times and measure the failure rate; separately, run a representative sample of other inputs to check whether this failure mode is specific to this input or widespread
- C. Check the API response for error codes that indicate server-side problems
- D. Compare the output distribution before and after the latest code deployment

**7. What is the most likely cause and how do you fix it?**

- A. The model has a memory of previous calls and is repeating itself
- B. The temperature is too low, concentrating probability on a few high-likelihood outputs; increase temperature to 0.7-1.0 to get more diverse options
- C. You need to include previous outputs in the prompt to explicitly ask for different ones
- D. API rate limiting is causing the model to fall back to cached responses

**8. What is the correct approach to handle unexpected model output types in a production system?**

- A. Trust the model to always return the correct type and do not add type checks
- B. Validate the output and raise an exception on type mismatch -- this forces the bug to be visible
- C. Validate the output; on failure, log the raw output for review, return a safe default, and continue processing rather than dropping the request
- D. Wrap the entire pipeline in a try/except that silently discards all errors

_Answers are in `checks.json` in the lesson directory._